[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chung-I/MiRA_training_course_2026/blob/main/notebooks/gnn_demo.ipynb)

# Graph Neural Networks — live demo notebook

MiRA Training Course 2026 · 9/3 · 仲翊

Click the badge above to open this in **Google Colab**. Nothing needs installing —
everything runs on **PyTorch and NumPy**, both of which Colab already has, and
**a CPU runtime is enough**. There is one optional PyTorch Geometric cell at the end
with its own install line.

Run the cells in order; each demo builds on the one before.

| # | Demo | The slide it backs |
|---|---|---|
| 1 | An MLP breaks when you renumber the nodes | "Flattening fails: permutation" |
| 2 | Message passing in 15 lines | "The message-passing equation" |
| 3 | L layers = an L-hop receptive field | "L layers = L-hop receptive field" |
| 4 | The graph carries the signal (GCN 91%, MLP 51%) | "Summarize your neighbors" |
| 5 | Building a k-NN graph on a point cloud | "90% of your time is graph construction" |
| 6 | Depth stops paying, then falls off a cliff | "Over-smoothing" |
| 7 | Relative positions on edges buy invariance | "Put the relative pose on the edge" |
| 8 | The same thing in PyTorch Geometric | "A GNN in 12 lines" |

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np

torch.manual_seed(0); np.random.seed(0)
print('torch', torch.__version__)

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:
    HAVE_MPL = False
    print('matplotlib not found - plots will be printed as numbers instead')

---
## Demo 1 — An MLP breaks when you renumber the nodes

A tabletop scene: 6 objects, each with 3 features. Nothing about the scene changes if
we write the objects down in a different order. An MLP on the flattened scene does not
know that.

In [ ]:
N, Fdim = 6, 3
X = torch.randn(N, Fdim)                     # 6 objects, 3 features each
A = torch.zeros(N, N)                        # who is next to whom
for i, j in [(0,1),(1,2),(2,3),(3,4),(4,5),(0,5),(1,4)]:
    A[i,j] = A[j,i] = 1.0

mlp = nn.Sequential(nn.Linear(N*Fdim, 32), nn.ReLU(), nn.Linear(32, 1))

perm = torch.randperm(N)                     # same scene, different order
print('permutation:', perm.tolist())

out_orig = mlp(X.reshape(1, -1))
out_perm = mlp(X[perm].reshape(1, -1))
print(f'MLP on the original order : {out_orig.item(): .4f}')
print(f'MLP after renumbering     : {out_perm.item(): .4f}')
print(f'--> difference            : {abs(out_orig.item()-out_perm.item()): .4f}   (should be 0, it is not)')

In [ ]:
# The same scene through one round of message passing, then a sum readout.
def message_passing_readout(X, A, W):
    H = (A @ X) @ W          # each node sums its neighbours, then a linear map
    return H.sum(0)          # order-independent readout

W = torch.randn(Fdim, 4)
r_orig = message_passing_readout(X, A, W)
r_perm = message_passing_readout(X[perm], A[perm][:, perm], W)   # permute rows AND cols

print('GNN on the original order :', r_orig.detach().numpy().round(4))
print('GNN after renumbering     :', r_perm.detach().numpy().round(4))
print(f'--> max difference        : {(r_orig-r_perm).abs().max().item():.2e}   (zero, up to float error)')

**The point:** permutation invariance is not something the GNN *learns*. It is built
into the architecture, so no training data is spent on it.

---
## Demo 2 — Message passing in 15 lines

    h_v  <-  UPDATE( h_v , AGG{ MSG(h_v, h_u) : u in N(v) } )

Written out for a whole graph at once, with a dense adjacency matrix.

In [ ]:
def normalize_adj(A):
    '''A_hat = D^-1/2 (A + I) D^-1/2  -- the GCN normalisation.'''
    A_hat = A + torch.eye(A.shape[0])
    deg = A_hat.sum(1)
    d_inv_sqrt = deg.pow(-0.5)
    return d_inv_sqrt.unsqueeze(1) * A_hat * d_inv_sqrt.unsqueeze(0)


class GCNLayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out)

    def forward(self, H, A_norm):
        return A_norm @ self.lin(H)      # AGG = normalised mean, UPDATE = linear


A_norm = normalize_adj(A)
layer = GCNLayer(Fdim, 4)
H1 = layer(X, A_norm)
print('node features in :', tuple(X.shape))
print('node features out:', tuple(H1.shape))
print()
print('Node 0 has neighbours', A[0].nonzero().flatten().tolist(),
      '- its new feature is a weighted mean of itself and those.')

---
## Demo 3 — L layers = an L-hop receptive field

Put a 1 on node 0 and nothing anywhere else, then propagate. After L rounds, exactly
the nodes within L hops are non-zero. This is the CNN receptive-field picture.

In [ ]:
signal = torch.zeros(N, 1); signal[0] = 1.0
h = signal.clone()
for L in range(1, 4):
    h = (A + torch.eye(N)) @ h
    reached = (h.flatten() > 0).nonzero().flatten().tolist()
    print(f'after {L} layer(s): node 0 can see nodes {reached}')

---
## Demo 4 — The graph carries the signal

Two communities, 200 nodes. **The node features are pure noise** — the only way to tell
the communities apart is who is connected to whom. So the MLP is stuck at chance and
the GCN is not.

In [ ]:
def two_community_graph(n=200, p_in=0.10, p_out=0.02, feat_dim=16):
    y = torch.cat([torch.zeros(n//2), torch.ones(n - n//2)]).long()
    P = torch.where(y[:,None] == y[None,:], torch.tensor(p_in), torch.tensor(p_out))
    A = (torch.rand(n, n) < P).float()
    A = torch.triu(A, 1); A = A + A.t()          # symmetric, no self-loops
    X = torch.randn(n, feat_dim)                 # features carry NO information
    return X, A, y

X2, A2, y2 = two_community_graph()
A2n = normalize_adj(A2)
n = X2.shape[0]
idx = torch.randperm(n); train_idx, test_idx = idx[:40], idx[40:]
print(f'{n} nodes, {int(A2.sum().item()//2)} edges, 40 labelled nodes for training')

In [ ]:
class GCN(nn.Module):
    def __init__(self, d_in, d_hid, d_out, n_layers=2):
        super().__init__()
        dims = [d_in] + [d_hid]*(n_layers-1) + [d_out]
        self.layers = nn.ModuleList([GCNLayer(dims[i], dims[i+1]) for i in range(n_layers)])

    def forward(self, H, A_norm, return_hidden=False):
        for i, layer in enumerate(self.layers):
            H = layer(H, A_norm)
            if i < len(self.layers) - 1:
                H = F.relu(H)
        return H


def train_node_classifier(model, uses_graph, epochs=200):
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        out = model(X2, A2n) if uses_graph else model(X2)
        F.cross_entropy(out[train_idx], y2[train_idx]).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        out = model(X2, A2n) if uses_graph else model(X2)
        return (out[test_idx].argmax(1) == y2[test_idx]).float().mean().item()


mlp_acc = train_node_classifier(nn.Sequential(nn.Linear(16,32), nn.ReLU(), nn.Linear(32,2)), False)
gcn_acc = train_node_classifier(GCN(16, 32, 2, n_layers=2), True)
print(f'MLP on features only : {mlp_acc*100:5.1f}%   (chance is 50%)')
print(f'2-layer GCN          : {gcn_acc*100:5.1f}%')

---
## Demo 5 — Building a k-NN graph on a point cloud

This is the step you will actually spend your time on. Same points, different `k`,
completely different graph.

In [ ]:
def make_point_cloud(n=150):
    t = torch.rand(n) * 2 * np.pi
    ring = torch.stack([torch.cos(t), torch.sin(t)], 1) * (1 + 0.05*torch.randn(n,1))
    blob = torch.randn(n//2, 2) * 0.12
    return torch.cat([ring, blob], 0)


def knn_graph(P, k):
    d = torch.cdist(P, P)
    d.fill_diagonal_(float('inf'))
    idx = d.topk(k, largest=False).indices          # k nearest for each point
    A = torch.zeros(len(P), len(P))
    A.scatter_(1, idx, 1.0)
    return torch.maximum(A, A.t())                  # make it symmetric


P = make_point_cloud()
for k in (3, 8, 20):
    A_k = knn_graph(P, k)
    print(f'k={k:2d}: {int(A_k.sum().item()//2):5d} edges, '
          f'mean degree {A_k.sum(1).mean().item():.1f}')

In [ ]:
if HAVE_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    for ax, k in zip(axes, (3, 8, 20)):
        A_k = knn_graph(P, k)
        for i, j in A_k.nonzero().tolist():
            if i < j:
                ax.plot(P[[i,j],0], P[[i,j],1], lw=0.4, color='0.65', zorder=1)
        ax.scatter(P[:,0], P[:,1], s=12, zorder=2)
        ax.set_title(f'k = {k}'); ax.set_aspect('equal'); ax.axis('off')
    plt.suptitle('Same points. Three different graphs. This choice is your model.')
    plt.tight_layout(); plt.show()
else:
    print('(install matplotlib to see the three graphs side by side)')

**Ask the room:** which one is right? There is no answer without the task. Too small
and the object falls apart into pieces; too large and everything touches everything and
you have thrown the structure away.

---
## Demo 6 — Depth stops paying, and then falls off a cliff

Train the same GCN at depths 1 to 8 and watch two numbers: test accuracy, and how
different the node embeddings still are from each other.

In [ ]:
depths, accs, spreads = list(range(1, 9)), [], []
for L in depths:
    torch.manual_seed(0)
    m = GCN(16, 32, 2, n_layers=L)
    accs.append(train_node_classifier(m, True))
    with torch.no_grad():
        H = X2
        for i, layer in enumerate(m.layers[:-1]):
            H = F.relu(layer(H, A2n))
        spreads.append(torch.pdist(H).mean().item())   # how different are the nodes?

for L, a, s in zip(depths, accs, spreads):
    print(f'{L} layers: test acc {a*100:5.1f}%   mean pairwise embedding distance {s:7.3f}')

In [ ]:
if HAVE_MPL:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.plot(depths, [a*100 for a in accs], 'o-'); a1.set_xlabel('layers')
    a1.set_ylabel('test accuracy (%)'); a1.set_title('Accuracy peaks at 2-3 layers')
    a2.plot(depths, spreads, 'o-', color='crimson'); a2.set_xlabel('layers')
    a2.set_ylabel('mean pairwise distance'); a2.set_title('Embeddings collapse together')
    plt.tight_layout(); plt.show()

**Read the second column first.** The mean pairwise distance between node embeddings
collapses toward zero as depth grows: after enough rounds of averaging with your
neighbours, every node in a connected graph ends up holding the same vector. That is
**over-smoothing**, and it is a property of the averaging itself, not of the task.

The accuracy column then does what you would expect: it climbs while extra hops still
add information, flattens, and drops to chance once the embeddings have collapsed.
Two honest caveats worth saying out loud:

- On this toy task the label is a *global* property (which community?), so extra hops
  keep helping longer than they would on a real dataset. On the standard node
  classification benchmarks the practical optimum is **2–3 layers**.
- Part of the collapse at large depth is plain optimization difficulty — deep GCNs
  without residual connections are hard to train. Both failures point the same way.

**The practical rule survives either explanation: start at 2 layers.** If you think you
need more, add a global node or a hierarchy rather than more rounds of averaging.

---
## Demo 7 — Put the relative position on the edge

Absolute coordinates as node features → move the whole scene one metre to the left and
the answer changes. Relative positions on the edges → it does not.

In [ ]:
def gnn_absolute(P, A):
    return ((A @ P) @ torch.ones(2, 1)).sum()          # uses absolute xy


def gnn_relative(P, A):
    total = 0.0
    for i, j in A.nonzero().tolist():
        total = total + (P[j] - P[i]).norm()           # uses only relative offsets
    return total


A_p = knn_graph(P, 8)
shift = torch.tensor([1.0, -0.5])
print(f'absolute-coord GNN, original : {gnn_absolute(P, A_p).item(): .4f}')
print(f'absolute-coord GNN, shifted  : {gnn_absolute(P + shift, A_p).item(): .4f}  <- changed')
print(f'relative-edge  GNN, original : {gnn_relative(P, A_p).item(): .4f}')
print(f'relative-edge  GNN, shifted  : {gnn_relative(P + shift, A_p).item(): .4f}  <- unchanged')

The same argument works for rotation, and it is why learned simulators generalise to
scenes they were never trained on. **Choose edge features that already have the
invariance you want, instead of paying for it in training data.**

---
## Demo 8 — The same thing in PyTorch Geometric (optional)

Everything above was written out by hand so it would run anywhere. In practice you
write this. On Colab, run the install line in the cell below first (about a minute);
skip the whole cell if you would rather not.

In [ ]:
# On Colab, uncomment to install (takes about a minute):
# !pip install -q torch_geometric

try:
    from torch_geometric.nn import GCNConv, global_mean_pool
    from torch_geometric.data import Data

    edge_index = A2.nonzero().t().contiguous()      # PyG wants a 2 x E edge list
    data = Data(x=X2, edge_index=edge_index, y=y2)

    class PyGGCN(nn.Module):
        def __init__(self, d_in, d_hid, d_out):
            super().__init__()
            self.c1, self.c2 = GCNConv(d_in, d_hid), GCNConv(d_hid, d_out)

        def forward(self, d):
            h = F.relu(self.c1(d.x, d.edge_index))
            return self.c2(h, d.edge_index)

    m = PyGGCN(16, 32, 2)
    opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(200):
        opt.zero_grad()
        F.cross_entropy(m(data)[train_idx], y2[train_idx]).backward()
        opt.step()
    with torch.no_grad():
        acc = (m(data)[test_idx].argmax(1) == y2[test_idx]).float().mean()
    print(f'PyG GCN test accuracy: {acc.item()*100:.1f}%')
except ImportError:
    print('torch_geometric is not installed - that is fine, everything above ran without it.')
    print('Install: pip install torch_geometric')

---
## What to take away

1. Permutation invariance and variable size come **free** with the architecture (Demo 1).
2. A GNN layer is a normalised neighbour average plus a linear map (Demo 2).
3. Layers are hops. Two or three is usually the right number (Demos 3 and 6).
4. When the structure carries the signal, the graph is the whole model (Demo 4).
5. **You will spend your time on Demo 5**, not on any of the others.

Where to go next: Stanford CS224W, the PyTorch Geometric examples directory, and the
Distill article *A Gentle Introduction to Graph Neural Networks*.